In [2]:
%reload_ext autoreload
%autoreload 2

from openai_api import ChunkingAgent, OpenAIAgent
import pandas as pd
import json
import os
from tqdm import tqdm
from pyvent import SaveTool

import nest_asyncio
nest_asyncio.apply()

# Example 1: FLO reddit summarization

In [3]:
from pyvent import SaveTool
from pyvent import ROOT_DIR

COMPANY = 'flo'

oai_saver = SaveTool(data_path=os.path.join(ROOT_DIR, 'data', COMPANY, 'oai'), input_path='input', output_path=f'output', 
                 cache_path='/tmp/', cache_name='cache')

## Import posts data

In [4]:
df_reviews = pd.read_csv(os.path.join(ROOT_DIR, 'data/tests/openai/chunking/reddit_post_comments.csv'))
df_reviews['comments'] = df_reviews['comments'].apply(lambda x: eval(x))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\MGadupudi\\PycharmProjects\\ImperialDadeCategoryManagement\\OpenAI Code\\data/tests/openai/chunking/reddit_post_comments.csv'

In [4]:
print(df_reviews.shape)

df_reviews.head(2)

(40, 7)


,title,text,subreddit,comments,url,upvotes,created_at
0,Dpo 15???,Hoping for some advice and support. \r\nI beli...,TFABLinePorn,"[{'level': 0, 'comment': 'My tests look simila...",https://www.reddit.com/gallery/1b7056a,1.0,2024-03-05
1,4 days late - Flo App,My period is 4 days late and I’m trying not to...,menstruation,"[{'level': 0, 'comment': 'Apps like flo are no...",https://www.reddit.com/r/menstruation/comments...,1.0,2023-09-16


### Parse Posts

In [5]:
review_format = """Post URL: {url}
Post Title: {title}
Post Text: {post_text}

Comments:
{comments_text}
"""
reviews_parsed = []
for i, row in df_reviews.iterrows():
    title = row['title']
    post_text = row['text']
    comments = row['comments']
    comments_text = ""
    for comment in comments:
        level = comment['level']
        text = comment['comment']
        text = text.replace('\n', ' ')
        if level == 0:
            comments_text += text + '\n'
        else:
            comments_text += '\t' * level + '#'*level + ' ' + text + '\n'
            
    review_text = review_format.format(url=row['url'], title=title, post_text=post_text, comments_text=comments_text)
    reviews_parsed.append(review_text)

In [6]:
print(reviews_parsed[0])

Post URL: https://www.reddit.com/gallery/1b7056a
Post Title: Dpo 15???
Post Text: Hoping for some advice and support. 
I believe I’m around 20dpo but starting to get concerned and hoping that isn’t accurate (tracked by flo). I did a test 1 week ago (26/2) and it was super faint but positive (image 4) I did a digital that same day and got a positive. I went to my dr and had bloods done 2 days lager and the level only came back at 36 (dr said only around 2 weeks, but given when last period was and when ovulation was meant to be that didn’t make sense). I had the test in image 3 the next morning (27/2) and then on Friday 01/03 i did the pink dye test, second in the stack was the day after so on. 
I did a digital with weeks on Saturday and it said 1-2, i did another today and it was still 1-2. 
Im due to go back for another blood test tomorrow and results on Thursday.
I guess im just looking for some advice or any support has anyone else had this? I’m worried it may mean a chemical as it d

## Summarize each post

**Only neccessary if your reviews / posts are very long**

In [ ]:
agent = OpenAIAgent(model='gpt-4o-mini', chunk_size=8, saver=oai_saver)
system_message = """
You will recieve a reddit post about the women's health App Flo and a tree of its associated commments where each level is indicated by the number of #s.
Your job is to provide an extensive summary of the post and following commentary.
You will include things such as the main points of the post, the main points of the discussion in the comments, and any other relevant information.
"""

user_prompts = reviews_parsed[:32]
prompts = agent.generate_prompts(system_prompts=system_message, user_prompts=user_prompts)
responses = agent.get_responses(prompts, temperature=0.0)

Running cost $0.0100: 100%|██████████| 4/4 [00:24<00:00,  6.05s/chunk]


## Start chunking

In [ ]:
system_prompt_layer_first = """
You will receive a list of summaries of reddit posts / their associated comments.
These posts will be about the women's health App Flo, please discard any posts that have nothing to do with this app.
"""
user_prompt_layer_first = """
Summarize the positives about the app in the following posts into bullet points.
Please include the most important points first.
Reviews: {reviews}
"""

system_prompt="You will recieve a list of positive bullet points regarding the Women's health app Flo."
user_prompt="""Please summarize the following summaries into bullet points.
Please include the most important points first.
Reviews: {reviews}
"""
system_prompt_layer_last="You will recieve a list of positive summaries regarding the Women's health app Flo. The bullets are sorted by importance, where the most important topics are at the top."
user_prompt_layer_last="""Please summarize the following reviews into exactly 8 bullet points.
Please include the most important points first.
Reviews: {reviews}
"""
# user_prompt_layer_first = user_prompt_layer_first.replace('positive','negative')
# user_prompt_layer_last = user_prompt_layer_last.replace('positive','negative')
# user_prompt = user_prompt.replace('positive','negative')
# system_prompt_layer_last = system_prompt_layer_last.replace('positive','negative')
# system_prompt = system_prompt.replace('positive','negative')
# system_prompt_layer_first = system_prompt_layer_first.replace('positive','negative')

agent = ChunkingAgent(
    model='gpt-4o-mini',
    batch_model='gpt-4o-mini-batch',
    chunk_size=32,
    system_prompt_layer_first=system_prompt_layer_first,
    user_prompt_layer_first=user_prompt_layer_first,
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    user_prompt_layer_last=user_prompt_layer_last,
    system_prompt_layer_last=system_prompt_layer_last,
    iter_size_first=8,
    iter_size=2,
    iter_size_last=2,
)

In [9]:
agent.iter_size * agent.iter_size_first * agent.iter_size_last

32

In [10]:
len(responses)

32

### Get Summaries

In [ ]:
# if you pass in more than 30,000 prompts it will automatically switch to batch (batch_trigger param)
summary = agent.get_summary(responses, temperature=0.0)

Running cost $0.0034: 100%|██████████| 1/1 [00:01<00:00,  1.80s/chunk]


In [12]:
# details of each layer of the chunking agent stored in the layers attribute
len(agent.layers)

4

In [13]:
print(summary)

- **Cycle Tracking Accuracy**: Flo is highly regarded for its effective tracking of menstrual cycles and ovulation, making it a valuable tool for those trying to conceive (TTC).
- **User-Friendly Interface**: The app features an intuitive design that simplifies navigation and tracking of menstrual health.
- **Community Support**: Users appreciate the supportive community that shares experiences and advice, fostering camaraderie, especially for those with irregular periods.
- **Comprehensive Tracking Features**: Flo offers extensive options for tracking various aspects of menstrual health, including hormonal changes, symptoms, and pregnancy.
- **Health Insights**: Users value the health and lifestyle tips shared within the community, including recommendations for supplements and healthy practices to enhance fertility.
- **Emotional Connection**: The app provides a platform for discussing emotional experiences related to menstrual health, PMS, and pregnancy, allowing for shared feelings 

In [14]:
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown(string))

In [15]:
levels = list(agent.layers.keys())
last_level = levels[1]

printmd(f'**Level {last_level}**')
for chunk in range(len(agent.layers[last_level])):
    printmd(f'**Chunk {chunk}**')
    printmd(f'{agent.layers[last_level][chunk]}')

**Level 1**

**Chunk 0**

### Positives about the Flo App from Reddit Posts:

- **Community Support**: Users express gratitude for the supportive community that shares experiences and advice related to early pregnancy and menstrual health, fostering a sense of camaraderie.
- **Tracking Features**: The app is recognized for its ability to track menstrual cycles, which can help users understand their bodies better and identify patterns over time.
- **Awareness of Cycle Variability**: Discussions highlight that the app can help users recognize that variations in menstrual cycles are normal, providing reassurance during times of uncertainty.
- **Resource for TTC (Trying to Conceive)**: Users mention using the app to track their cycles while trying to conceive, indicating its utility in fertility awareness.
- **Emotional Connection**: The app facilitates discussions about the emotional aspects of menstrual health, allowing users to share their feelings and experiences related to PMS and pregnancy.
- **Encouragement for Healthy Practices**: Users share advice on lifestyle changes and health practices that can be tracked using the app, such as taking prenatal vitamins and monitoring symptoms.

These points reflect the positive aspects of the Flo app as experienced by users in the context of community support and health tracking.

**Chunk 1**

### Positives about the Flo App from Reddit Posts:

- **Cycle Tracking Accuracy**: Users express interest in the Flo app's ability to track menstrual cycles and ovulation, indicating it can be a helpful tool for those trying to conceive (TTC).
- **Community Support**: The app is part of a broader community where users share experiences and advice, fostering a supportive environment for individuals navigating fertility and reproductive health.
- **Integration with Other Tools**: Some users mention using the Flo app alongside ovulation predictor kits (OPKs) for more accurate tracking, suggesting that it can complement other fertility awareness methods.
- **Emotional Connection**: The app serves as a platform for users to discuss their emotional journeys related to conception, providing a space for sharing both concerns and successes.
- **Health Insights**: Users appreciate the health and lifestyle tips shared within the community, which often include recommendations for supplements and lifestyle changes to improve fertility.
- **User Engagement**: The app encourages users to engage with their health data and seek advice, which can empower them in their TTC journey.

**Chunk 2**

- **User-Friendly Features**: Flo is praised for its accuracy and ease of use, allowing users to edit period dates and gain insights into their menstrual patterns.
- **Long-Term Satisfaction**: Many users have been using Flo for years and express satisfaction with the free version of the app.
- **Community Support**: The app fosters a supportive community where users share personal experiences and practical advice for tracking menstrual cycles, especially for those with irregular periods.
- **Comprehensive Tracking**: Flo offers options to track various aspects of menstrual health, including hormonal changes and symptoms, which can be beneficial for users managing their cycles post-IUD removal or while trying to conceive.
- **Privacy Concerns Addressed**: While some users express frustration with ads in Flo, others appreciate its features and the ability to track their health without needing to switch to other apps.

**Chunk 3**

Here are the positives about the Flo app mentioned in the relevant posts:

- **User-Friendly Interface**: Users appreciate Flo for its intuitive design, making it easy to navigate and track menstrual cycles.
- **Customizable Symptom Tracking**: The app allows users to track specific symptoms, which is particularly beneficial for those with conditions like endometriosis.
- **Comprehensive Features**: Flo offers a range of features, including pregnancy tracking and ovulation predictions, which many users find helpful.
- **Community Support**: The app fosters a sense of community among users, providing a platform for sharing experiences and advice related to menstrual health.
- **Educational Resources**: Users find value in the educational content provided by the app, which helps them understand their bodies and menstrual cycles better.
- **Privacy Considerations**: Some users appreciate Flo's approach to data privacy compared to other apps, although there are mixed feelings about privacy requirements.
- **Integration with Other Tracking Methods**: Users mention that Flo can be effectively used alongside other methods, such as ovulation predictor kits (OPKs) and basal body temperature (BBT) tracking, enhancing its utility.

## Get relevant posts w/ Embeddings

In [16]:
# Pyvent embeddings tool
api = OpenAIAgent()

##### Create Embeddings DF from reviews

In [17]:
text_list = []
for i, row in df_reviews.iterrows():
    url = row['url']
    title = row['title']
    text = row['text']
    text = f"""{row['title']} {row['text']}"""
    date = row['created_at']
    text_list.append(
        {
            'url': url,
            'text': text,
            'date': date
        }
    )
    for comment in row['comments']:
        text_list.append(
            {
                'url': url,
                'text': comment['comment'],
                'date': date
            }
        )

In [18]:
# get batches of 100

embeddings_list = []
chunk_size = 100
text_list = text_list[:5000]
for chunk in tqdm(range(0, len(text_list), chunk_size)):
    chunk_list = text_list[chunk:chunk+chunk_size]
    chunk_text_list = [item['text'] for item in chunk_list]
    data = api.embeddings(chunk_text_list)
    embeddings_list.extend(data)

100%|██████████| 11/11 [00:28<00:00,  2.57s/it]


In [19]:
print(len(embeddings_list))

for text, embedding in zip(text_list, embeddings_list):
    text['embedding'] = embedding
    
df_embeddings = pd.DataFrame(text_list)
df_embeddings['date'] = pd.to_datetime(df_embeddings['date'])
df_embeddings['year'] = df_embeddings['date'].dt.year

print(df_embeddings.shape)
df_embeddings.head(2)

1055
(1055, 5)


,url,text,date,embedding,year
0,https://www.reddit.com/gallery/1b7056a,Dpo 15??? Hoping for some advice and support. ...,2024-03-05,"[-0.01351909339427948, 0.025286251679062843, -...",2024
1,https://www.reddit.com/gallery/1b7056a,My tests look similar at 14dpo. How did your b...,2024-03-05,"[0.010819533839821815, 0.013368776068091393, -...",2024


In [20]:
# Embed your query
query = "Flo is a great app"
query_embedding = api.embeddings(query)

In [21]:
# calculate similarity between all posts
df_embeddings['sim'] = df_embeddings['embedding'].apply(lambda x: api.cosine_similarity(query_embedding, x))

In [22]:
# sort by sim descending and get top 10
sim_texts = df_embeddings.sort_values(by='sim', ascending=False).head(10)

In [23]:
# print the text and url
for i, row in sim_texts.iterrows():
    print(row['text'])
    print(row['url'])
    print('\n')
    break

I like using Flo
https://www.reddit.com/r/PCOS/comments/1bc40ju/looking_for_an_easy_periodtracking_app_not_clue/




# Example 2: review summary

In [24]:
reviews = [
    "Bought this trolley because I have to move to a different state for my work. I was looking for something that'll give me some features(Polycarbonate/TSA Lock/8 wheels were my priority). Since I'd be travelling alone I was looking for a medium size suitcase that'll be spacious enough to accomodate most of my stuff. Got the delivery with one day. \nSo far so good. Zip good, colour good, material good, handle good, lock good, wheels good, inside of the bag good. \nI'll try to update my review after use. I have included all the pics from most of the angles.\nDon't think much. Don't look at the negative comments. Every goddam* product will have negative comments so don't get discouraged by that. I'd suggest you but anything that has 4*+ rating not just overall but in every field of the product and Also the seller rating. 5* rating for me in this product.",
    "I just happened to buy this from one of the Vendor Outside flip kart and found to be an amazing piece. It seems that VIP has ensured to check every quality point check and build a very sturdy Suitcase.  \nAll four wheels are well connected to each other Internally making it very strong wheel base\nHas provided a handle at the bottom making it very convenient to hold the luggage while its heavy\nThe cloth material is a top of the water resistant. The water just flows away not allowing even a singly drop of water to be absorbed by the cloth.\nIt comes in 3 amazing colour, where Maroon & Blue is outclass.\nWould recommend to everyone who are looking to buy this product. It won't disappoint you.",
    "The fabric is of good quality, price point was good, delivery was quite delayed. The packaging from seller was bad. Received an used product as there were dust marks. The wheels and the trolley puller is of decent material. This won't be suitable for rough use. Bag is not sturdy and can't withstand heavy load. It is suitable for for 15kg of check-in luggage as there is enough space. The sides could have had polycarbonate backing for sturdy use. Only corners now have bit sturdy back. Colour is as shown in the images. Overall a decent product",
    'This bag got completely crushed from the top during some kind of airport handling. It managed only one trip for us, and in the second one, it got crushed. My family and I travel a lot internationally and all our checkin suitcases are from VIP, however this is the only one that has disappointed me so far. Also , please be careful about the dimension. Some airlines can be sensitive about baggage dimensions, and this one exceeds the standard dimensions.',
    'I recently received this VIP stargaze Active 55 small cabin suitcase.This luggage bag is very strong and light in weight so very easy to use and very smooth wheels so hassle free movement. I am really very happy with this trolley bag because this bag  exactly fits my expectations for a perfect trolley bag. It has plenty of space and multiple compartments so keep things accordingly.Main part is attractive designs and bold colours. and for security purposes combination lock is provided. Totally fantastic product and value for money.',
    'I definitely loved how this bag looks. Its design is simple yet elegant. Also its easy to handle, wheels, zips and all handles are so smoothly. I ordered a small size and it is spacious which makes it perfect for your trips. Proper partition is done so you can organise your things properly. So happy with the purchase .It is value for money. Skybags are good. i also have its office bag. They offer premium designs which look great on day to day purposes. build quality is great',
    'This lightweight suitcase fullfills all mine need. It comes with Spinner whleels that can be rotate on 360 degree so very easy to transport and very durable. All my personal belongings are organized easily due to multiple compartments in it. This trolly type suitcase looks very effective and impressive due to its stylish design. Its inner design is also amazing and lengthy with multiple zip compartment. In short a special designed trolly type suitcase for me.',
    "It's cute and perfect for frequent travel. The look of the bag can be deceiving, though the size is small from outside, it has so much space for clothes and other accessories. You can fit in more clothes than you except. The lock is really good and strong. Definitely worth the price.It has great stability. You can even pace or run with this bag. It moves all sides. Easy to use. The performance is great .",
    'Suitcase has exceeded my expectations. The hard shell is durable and provides excellent protection for my belongings while the TSA lock gives me peace of mind during travel. The suitcase glides effortlessly in any direction making it incredibly easy to navigate through busy airports or train stations.The handle is sturdy and adjustable, making it comfortable to carry regardless of my height',
    'VIP Trolley bag is more than expected.\nTSA Lock\nEight Wheel\nMultiple Inner Pockets with wet pocket\nSmooth runners\nLight weight polypropylene shell\nHeavy three step aluminum handle\nBody coloured accessories fited.\nScratchless body due to mat finish\n   For me it is a nice deal. Perfect for International and Domestic travelling.',
    'Go for it its an old brand vip trustable brands of all ours indian people without hassle go for it its has 8 wheels thats an big nd great things of this product better then American tourister sky bag nd safari best quality with value for money nd quality too all rounder its good nd best i wil say',
    'I got it on sale. paid around 6.5k for both the sizes.\n\nIts not the large and medium, the seller hasnt mentioned proper size in the description.\nIts worth the buy, its got all the fetures like\n1. TSA Lock\n2. Anti theft zipp, (double zip from inside and out )\n3. dual spinner wheels\n4. international warrenty',
    "After searching and doing deep analysis of more than 50 products from top 4-5 brands I've found this product and I can say I'm very happy with it's quality and specifications.\n\nI was looking for \n\nTSA LOCK\nANTI THEFT ZIP\n8 WHEELS\nSUPERIOR BUILD QUALITY \n\nand it has all these. At this budget you can blindly go for it.",
    'This trolley bag is designed for those who value space and functionality. Its spacious interior allows for organized packing, while the lightweight build makes it easy to carry. Multiple compartments help keep items in order, and smooth wheels ensure hassle-free movement, making it a practical choice for travel.',
    'Big billion day sales are so awesome..got this sweet suitcase for 3100. It is so roomy you guys! It can fit a 7 year old child inside.(not that you should) \n\nIt is huge and exactly what i wanted. Has double zip and double wheels that make it more durable than single wheels. Totally satisfied',
    'Polycarbonate is better than Polysilicon and ABS plastic. This one is polycarbonate. It has TSA lock so not to worry for international travel.\nElegant looks. \n\nI cant find any cons, except for fluctuating price. So keep a watch for right price.\n\nElse from quality perspective i am more than happy.'
] * 2

df_1 = pd.DataFrame(reviews, columns=['text'])

In [25]:
agent = ChunkingAgent(
    model = "gpt-4o-mini",
    iter_size_first=8,
    iter_size=2
)

In [26]:
num_words_per_bullet = 20
num_bullet_points = 10

system_message_1 = f"""
Problem: You will receive several reviews on a product. Each review is numbered. Your job is to identify the **important** positive and negative topics that are mentioned across the reviews and summarize each topic around {num_words_per_bullet} words if possible. You should also count the number of reviews that mentions each topic. Return as many topics as you can identify but no more than {num_bullet_points} positive topics and {num_bullet_points} negative topics.
Solution:
Step 1: Identify core topics from each review. Group them into positive and negative topics.
Step 2: Find the most **important** common topics across reviews. The topics you identify might not be the most commonly mentioned ones but what you think is important for us to know about the product. However, each topic should be mentioned at least once (count should not be zero). Try to identify topics that are distinctive and try to be specific. Avoid general topics like 'good product', 'high quality' or 'works well', as those will likely be the same for similar products. Be specific and avoid over-generalization. If there are specific details that are important, try to include them. For example, instead of 'good for skin', you might say 'good for sensitive skin'; instead of 'taste issues', you might say 'taste is too sweet for some and strong for others'. Group similar topics, e.g. 'expensive' and 'a bit pricey' should be under the same topic. You can use longer text to descrive the topic, e.g. 'Expensive / pricey compared to competitors'.
Step 3: Count the number of reviews that mention each topic. If a topic is mentioned multiple times in the same review, count it only once.
Step 4: Return your output in json. 

Final Answer: Here's an example of how your output might look like:
{{
    'positive':[{{'topic': 'Plant based / Vegan friendly', 'count': X}}, {{'topic': 'Good for sensitive skin', 'count': X}}, {{'topic': 'Good scent / scent not too strong', 'count': X}}],
    'negative':[{{'topic': 'Expensive', 'count': X}}, {{'topic': 'Not good for oily skin', 'count': X}}, {{'topic': 'Packaging is too big', 'count': X}}]
}}
The above topics are placeholders for the topics you identify (and may be completely irrelevant to the reviews you will receive). X is a placeholder for the integer value representing the total number of reviews that mentioned that topic. 
"""

system_message_2 = f"""
Problem: You will receive extracts of reviews on a product. Each summary consist of the top positive and negative topics that are mentioned in the reviews and the count of reviews that mentioned that topic. Here's an example of how your an extract might look like:
{{
    'positive':[{{'topic': 'Plant based', 'count': X}}, {{'topic': 'Good for sensitive skin', 'count': X}}, {{'topic': 'Scent not too strong', 'count': X}}],
    'negative':[{{'topic': 'Expensive', 'count': X}}, {{'topic': 'Not good for oily skin', 'count': X}}, {{'topic': 'Packaging is too big', 'count': X}}]
}}
Your job is to identify the **important** positive and negative topics that are mentioned in the extracts and summarize each topic under {num_words_per_bullet} words if possible. You should also tally up the count of reviews that mention that topic.  Return as many topics as you can identify but no more than {num_bullet_points} positive topics and {num_bullet_points} negative topics.

Solution:
Step 1: Identify the common positive and negative topics across the extracts. For example, if 'Plant based' is in extract 1 and 'Vegan friendly' is in extract 2, you could group them together as 'Plant based / Vegan friendly'. The topics you identify might not be the most commonly mentioned ones but what you think is important for us to know about the product. Try to identify topics that are distinctive. Avoid general topics like 'good product', 'high quality' or 'works well', as those will likely be the same for similar products. Be specific and avoid over-generalization. If there are specific details that are important, try to include them. For example, instead of 'good for skin', you might say 'good for sensitive skin'; instead of 'taste issues', you might say 'taste is too sweet for some and strong for others'.
Step 2: Add up the number of reviews that mention each topic. For example, if in extract 1 'Plant based' has a count of 3, in extract 2 'Vegan friendly' has a count of 2, in extract 3 'Suitable for vegans' has a count of 4, you should return 'Plant based / Vegan friendly' with a count of 3+2+4=9.
Step 3: Return your output in json.

Final Answer: Here's an example of how your output might look like:
{{
    'positive':[{{'topic': 'Plant based / from natural ingredients', 'count': X}}, {{'topic': 'Good for sensitive skin', 'count': X}}, {{'topic': 'Pleasant scent / Scent not too strong', 'count': X}}],
    'negative':[{{'topic': 'Expensive / Pricy compared to competitors', 'count': X}}, {{'topic': 'Not good for oily skin', 'count': X}}, {{'topic': 'Packaging is too big', 'count': X}}]
}}
The above topics are placeholders for the topics you identify (and may be completely irrelevant to the reviews you will receive). X is a placeholder for the integer value representing the total number of reviews that mentioned that topic. 
"""

In [27]:
agent.system_prompt_layer_first = system_message_1
agent.system_prompt = system_message_2
agent.system_prompt_layer_last = system_message_2

In [28]:
response = agent.get_summary(reviews, temperature=0.0, response_format={'type':'json_object'})
response = json.loads(response)

Running cost $0.0023: 100%|██████████| 1/1 [00:04<00:00,  4.66s/chunk]


In [29]:
ln = '\n'
topics = [x['topic'] for x in response['positive'] + response['negative']]

system_message_3 = f"""You will receive a review on a product. These are the top topics identified from the reviews of this product. Your job is to decide which one of these topics are mentioned in the review. Here are the topics:
{ln.join(topics)}

Return in comma separated values the topics that are mentioned in this review from the list above. Do not include any other topics or reword any of the topics. If none of the topics are mentioned, return 'None'.
"""

user_prompt = '{text}'

df_1 = agent.format_df_prompts(df_1,system_message_3,user_prompt)
df_1 = agent.run_df_prompts(df_1)

Running cost $0.0027: 100%|██████████| 1/1 [00:01<00:00,  1.60s/chunk]


In [30]:
pd.set_option('display.max_colwidth', 100)
df_1[['text','openai_response']]

,text,openai_response
0,Bought this trolley because I have to move to a different state for my work. I was looking for s...,"Lightweight and spacious design, Good security features with TSA lock, Elegant polycarbonate mat..."
1,I just happened to buy this from one of the Vendor Outside flip kart and found to be an amazing ...,"Lightweight and spacious design, Durable wheels with smooth movement / 360-degree rotation, Wate..."
2,"The fabric is of good quality, price point was good, delivery was quite delayed. The packaging f...","Value for money, Delayed delivery and poor packaging, Received used product with dust marks, Whe..."
3,This bag got completely crushed from the top during some kind of airport handling. It managed on...,"Crushed during airport handling, Exceeds standard baggage dimensions"
4,I recently received this VIP stargaze Active 55 small cabin suitcase.This luggage bag is very st...,"Lightweight and spacious design, Durable wheels with smooth movement / 360-degree rotation, Good..."
5,I definitely loved how this bag looks. Its design is simple yet elegant. Also its easy to handle...,"Stylish appearance with attractive colors, Lightweight and spacious design, Durable wheels with ..."
6,This lightweight suitcase fullfills all mine need. It comes with Spinner whleels that can be rot...,"Lightweight and spacious design, Durable wheels with smooth movement / 360-degree rotation, Styl..."
7,"It's cute and perfect for frequent travel. The look of the bag can be deceiving, though the size...","Lightweight and spacious design, Good security features with TSA lock, Value for money, Durable ..."
8,Suitcase has exceeded my expectations. The hard shell is durable and provides excellent protecti...,"Durable wheels with smooth movement / 360-degree rotation, Good security features with TSA lock,..."
9,VIP Trolley bag is more than expected.\nTSA Lock\nEight Wheel\nMultiple Inner Pockets with wet p...,"Good security features with TSA lock, Durable wheels with smooth movement / 360-degree rotation,..."


In [31]:
print(f'Review: \n{df_1.iloc[0]["text"]}')
print(f'\nTopics: \n{df_1.iloc[0]["openai_response"]}')

Review: 
Bought this trolley because I have to move to a different state for my work. I was looking for something that'll give me some features(Polycarbonate/TSA Lock/8 wheels were my priority). Since I'd be travelling alone I was looking for a medium size suitcase that'll be spacious enough to accomodate most of my stuff. Got the delivery with one day. 
So far so good. Zip good, colour good, material good, handle good, lock good, wheels good, inside of the bag good. 
I'll try to update my review after use. I have included all the pics from most of the angles.
Don't think much. Don't look at the negative comments. Every goddam* product will have negative comments so don't get discouraged by that. I'd suggest you but anything that has 4*+ rating not just overall but in every field of the product and Also the seller rating. 5* rating for me in this product.

Topics: 
Lightweight and spacious design, Good security features with TSA lock, Elegant polycarbonate material, Durable wheels with